# Notebook 6b: Rejecting Invented Values

Notebook 6 found that hybrid lost to vector-only, and that the SQL path failed because the model adds filters nobody asked for: `'Jim Clark'` in a question about the 2019 Italian Grand Prix, `season = 2023` in a career question. Those queries run without error, so the retry loop never sees them.

The fix is a Python check: every name and number in the SQL must come from the question, apart from a short safe list. Rejected queries go back through the retry, and if they still fail, `ask()` falls back to retrieval.

The code now lives in `src/`, and the checks are in `tests/test_literal_guard.py`.

In [ ]:
import json
import re
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import src.text_to_sql as text_to_sql
from src.router import answer_vector, ask
from src.text_to_sql import con, invented_values, numbers_in

with open(PROJECT_ROOT / "eval" / "questions.json", encoding="utf-8") as f:
    TEST4 = [q for q in json.load(f) if q["split"] == "test4"]

print(f"test4 questions: {len(TEST4)}")

## What the check can and can't see

It checks values, not meaning. A query using only the question's own words can still answer the wrong question.

In [ ]:
question = "Who started on pole at the 2019 Italian Grand Prix?"
copied = "SELECT COUNT(*) FROM f1_results WHERE driver_name = 'Jim Clark' AND grid = 1"
print(invented_values(copied, question))

# A wrong query that only uses the question's own words gets through
question = "How many Grands Prix did Jim Clark win?"
counts_entries = "SELECT COUNT(DISTINCT race_name) FROM f1_results WHERE driver_name = 'Jim Clark'"
print(invented_values(counts_entries, question))

## Evaluation

The `test4` questions were written after this notebook's code was committed. Each question runs once: at `temperature=0` the model is deterministic, so repeats would be identical, and more questions tell us more.

Each answer is right, wrong or refused. A wrong answer is the worst result, because it looks true.

In [ ]:
REFUSAL = re.compile(r"(not|isn't) in the (context|database|data)|no usable result|no information|"
                     r"cannot (provide|answer|determine)|can't answer|not mentioned|no .{0,30} mentioned|"
                     r"returned no rows", re.IGNORECASE)


def is_right(q: dict, answer: str) -> bool:
    values = q.get("accept", [])
    if q["truth_sql"]:
        values = [con.execute(q["truth_sql"]).fetchone()[0]] + values
    for value in values:
        if isinstance(value, str) and value.lower() in answer.lower():
            return True
        if not isinstance(value, str) and any(abs(n - float(value)) <= 0.01 for n in numbers_in(answer)):
            return True
    return False


def outcome(q: dict, answer: str) -> str:
    """right, refused, or wrong. A wrong answer is the worst result, because it looks true."""
    refused = bool(REFUSAL.search(answer))
    if q["truth_sql"] is None and not q.get("accept"):
        return "right" if refused else "wrong"
    if refused:
        return "refused"
    return "right" if is_right(q, answer) else "wrong"

In [ ]:
def hybrid(question: str, guard: bool) -> tuple:
    text_to_sql.USE_VALUE_GUARD = guard
    result = ask(question)
    text_to_sql.USE_VALUE_GUARD = True
    return result["answer"], result["path"]


rows = []
for q in TEST4:
    vector = answer_vector(q["question"])[0]
    no_guard, no_guard_path = hybrid(q["question"], guard=False)
    guarded, guarded_path = hybrid(q["question"], guard=True)
    rows.append({"id": q["id"], "kind": q["kind"], "question": q["question"],
                 "vector": outcome(q, vector), "hybrid_no_guard": outcome(q, no_guard), "hybrid_guard": outcome(q, guarded),
                 "path_no_guard": no_guard_path, "path_guard": guarded_path,
                 "vector_answer": vector, "no_guard_answer": no_guard, "guard_answer": guarded})

for name in ("vector", "hybrid_no_guard", "hybrid_guard"):
    counts = {k: sum(r[name] == k for r in rows) for k in ("right", "wrong", "refused")}
    print(f"{name:<16} {counts}")
for r in rows:
    print(f"{r['id']} {r['kind']:<13} V={r['vector']:<8} off={r['hybrid_no_guard']:<8} on={r['hybrid_guard']:<8} {r['path_guard']:<11} {r['guard_answer'][:80]!r}")

out = PROJECT_ROOT / "eval" / "results_nb06b.json"
out.write_text(json.dumps(rows, indent=2, ensure_ascii=False), encoding="utf-8")

## What We Learned

Scoring the final text on 20 `test4` questions, written after the code was committed:

| Pipeline | Right | Wrong | Refused |
|---|---|---|---|
| Vector only | 12 | 2 | 6 |
| Hybrid, guard off | 13 | 6 | 1 |
| Hybrid, guard on | 14 | 4 | 2 |

On the 12 fact questions, hybrid with the guard got 8 right and vector got 4. In Notebook 6 both got 2 of 8. All pipelines got the 3 narrative questions right.

### Findings

1. **Hybrid now beats vector-only on right answers,** which it didn't in Notebook 6. The two pipelines fail differently: vector refuses when unsure, and hybrid answers.
2. **The guard removes some wrong answers at a cost.** Wrong answers fell from 6 to 4, but one right answer (Ferrari's 2002 wins) became a refusal.
3. **Prompt edits were whack-a-mole.** Each one fixed a failure and caused another, which is why the fix moved into Python.
4. **Values aren't meaning.** "Jackie Stewart won 12 races" and "McLaren employs 53 distinct mechanics" pass, because those queries only use the question's own words.
5. **Fact-shaped unanswerable questions are the main gap.** Vector refused all 5 unanswerable questions, while hybrid gave invented answers to 2.

### Limits of this evaluation

- 20 questions, so a gap of two isn't conclusive.
- At `temperature=0` the model is deterministic, so each question ran once.
- test4_06 ("Seven races were held in 1950") is correct but scored wrong, because the scorer looks for the digit.
- test4_07 is scored right but is a raw list of winners with Rosberg first, so it shouldn't count.
- Those two errors cancel out for hybrid. The recorded scores are unchanged.
- `test` to `test4` have all been seen now, so the next change needs a new test set.
